# 📈 CareFlow — Notebook 5: Time-Series Forecasting
## ARIMA · Prophet · XGBoost Ensemble · ED Volume Forecasting

**Objectives:**
- Stationarity testing (ADF, KPSS)
- ARIMA model fitting with auto-selection
- Facebook Prophet seasonal decomposition
- XGBoost with lagged features
- Ensemble forecast with confidence intervals

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print('✅ Prophet available')
except ImportError:
    PROPHET_AVAILABLE = False
    print('⚠️  Prophet not installed. Install: pip install prophet')

plt.style.use('dark_background')
COLORS = ['#06b6d4','#8b5cf6','#10b981','#f59e0b','#f43f5e','#3b82f6']
print('✅ Environment ready')

In [ ]:
# ─── 1. Generate Realistic ED Daily Volume Time Series ────────────────────────
np.random.seed(42)

# 12 months of daily ED visit data (Jan–Dec 2026)
dates = pd.date_range(start='2026-01-01', end='2026-12-31', freq='D')
n = len(dates)

# Components:
trend = np.linspace(38, 52, n)  # slight upward trend (capacity growth)
day_of_week = np.array([1.0 if d.weekday() in [5,6] else 0.72 for d in dates])  # weekend surge
seasonal = 6 * np.sin(2 * np.pi * np.arange(n) / 365 + np.pi)  # winter peak, summer trough
noise = np.random.normal(0, 4.2, n)

# Special events
volume = (trend * day_of_week + seasonal + noise).round(0).astype(int)
# Flu spike in Jan–Feb
volume[:60] = (volume[:60] * 1.18).astype(int)
# Summer lull Jul–Aug
volume[180:243] = (volume[180:243] * 0.88).astype(int)

ts = pd.Series(volume, index=dates, name='daily_visits')

plt.figure(figsize=(18, 4))
plt.plot(ts.index, ts.values, color='#06b6d4', linewidth=1.5, alpha=0.9)
plt.fill_between(ts.index, ts.values, alpha=0.15, color='#06b6d4')
plt.title('ED Daily Visit Volume — 2026 (Jan–Dec)', fontsize=13, color='white')
plt.ylabel('Daily Visits', color='white')
plt.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../docs/ts_fig1_raw_series.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Series: {len(ts)} days | Mean: {ts.mean():.1f} | Std: {ts.std():.1f} | Min: {ts.min()} | Max: {ts.max()}')

In [ ]:
# ─── 2. Stationarity Testing ──────────────────────────────────────────────────
print('═'*60)
print('STATIONARITY TESTS')
print('═'*60)

# ADF Test
adf_result = adfuller(ts, autolag='AIC')
print(f'\nAugmented Dickey-Fuller (ADF) Test:')
print(f'  ADF Statistic: {adf_result[0]:.4f}')
print(f'  p-value:       {adf_result[1]:.4e}')
print(f'  Critical values: {adf_result[4]}')
print(f'  Result: Series is {"STATIONARY" if adf_result[1] < 0.05 else "NON-STATIONARY"}')

# KPSS Test
kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(ts, regression='c', nlags='auto')
print(f'\nKPSS Test (H0: stationary):')
print(f'  KPSS Statistic: {kpss_stat:.4f}')
print(f'  p-value:        {kpss_p:.4f}')
print(f'  Result: {"Reject H0 → NON-STATIONARY" if kpss_p < 0.05 else "Fail to reject H0 → STATIONARY"}')

# First difference if non-stationary
ts_diff = ts.diff().dropna()
adf_diff = adfuller(ts_diff)
print(f'\nAfter 1st Differencing:')
print(f'  ADF p-value: {adf_diff[1]:.4e} → {"STATIONARY" if adf_diff[1] < 0.05 else "NON-STATIONARY"}')

In [ ]:
# ─── 3. ACF / PACF Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('ACF and PACF Analysis', fontsize=13, fontweight='bold', color='white')

plot_acf(ts, lags=40, ax=axes[0,0], color='#06b6d4')
axes[0,0].set_title('ACF — Original Series', color='white')
axes[0,0].tick_params(colors='white')

plot_pacf(ts, lags=40, ax=axes[0,1], color='#8b5cf6', method='ywm')
axes[0,1].set_title('PACF — Original Series', color='white')
axes[0,1].tick_params(colors='white')

plot_acf(ts_diff, lags=40, ax=axes[1,0], color='#10b981')
axes[1,0].set_title('ACF — 1st Differenced Series', color='white')
axes[1,0].tick_params(colors='white')

plot_pacf(ts_diff, lags=40, ax=axes[1,1], color='#f59e0b', method='ywm')
axes[1,1].set_title('PACF — 1st Differenced Series', color='white')
axes[1,1].tick_params(colors='white')

plt.tight_layout()
plt.savefig('../docs/ts_fig2_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ACF/PACF plots saved')
print('  → ACF tails off slowly, PACF cuts off at lag 1-7 → Suggests ARIMA(p,1,q) orders')

In [ ]:
# ─── 4. Seasonal Decomposition ────────────────────────────────────────────────
decomp = seasonal_decompose(ts, model='additive', period=7)  # weekly seasonality

fig, axes = plt.subplots(4, 1, figsize=(18, 12))
fig.suptitle('Seasonal Decomposition (Additive, Period=7 days)', fontsize=13, fontweight='bold', color='white')

for ax, data, label, col in zip(axes,
    [ts, decomp.trend, decomp.seasonal, decomp.resid],
    ['Observed', 'Trend', 'Seasonal', 'Residual'],
    COLORS[:4]):
    ax.plot(data.index, data.values, color=col, linewidth=1.5)
    ax.set_ylabel(label, color='white', fontsize=10)
    ax.tick_params(colors='white')
    if label == 'Residual':
        ax.axhline(0, color='white', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../docs/ts_fig3_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Seasonal range: {decomp.seasonal.min():.1f} to {decomp.seasonal.max():.1f} visits')
print(f'Trend range: {decomp.trend.dropna().min():.1f} to {decomp.trend.dropna().max():.1f} visits')
print(f'Residual std: {decomp.resid.std():.2f} visits (random noise component)')

In [ ]:
# ─── 5. ARIMA Model ───────────────────────────────────────────────────────────
# Train/test split: last 30 days as test set
train = ts[:-30]
test = ts[-30:]

print('Fitting ARIMA(2,1,2) model...')
arima_model = ARIMA(train, order=(2,1,2), seasonal_order=(1,0,0,7))
arima_fit = arima_model.fit()
print(arima_fit.summary().tables[0])

# Forecast
forecast_result = arima_fit.get_forecast(steps=30)
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.10)  # 90% CI

arima_mae = mean_absolute_error(test, forecast_mean)
arima_rmse = np.sqrt(mean_squared_error(test, forecast_mean))
arima_mape = np.mean(np.abs((test - forecast_mean) / test)) * 100

print(f'\nARIMA Forecast Performance (30-day holdout):')
print(f'  MAE:  {arima_mae:.2f} visits')
print(f'  RMSE: {arima_rmse:.2f} visits')
print(f'  MAPE: {arima_mape:.2f}%')

In [ ]:
# ─── 6. XGBoost with Lagged Features ─────────────────────────────────────────
# Create lagged features for ML-based forecasting
def create_lag_features(series, n_lags=14):
    df_feat = pd.DataFrame({'y': series})
    for lag in range(1, n_lags+1):
        df_feat[f'lag_{lag}'] = df_feat['y'].shift(lag)
    df_feat['day_of_week'] = series.index.dayofweek
    df_feat['month'] = series.index.month
    df_feat['is_weekend'] = (series.index.dayofweek >= 5).astype(int)
    df_feat['rolling_7d'] = df_feat['y'].shift(1).rolling(7).mean()
    df_feat['rolling_14d'] = df_feat['y'].shift(1).rolling(14).mean()
    return df_feat.dropna()

lag_df = create_lag_features(ts)
X_lag = lag_df.drop(columns='y')
y_lag = lag_df['y']

# Split
split = len(X_lag) - 30
X_train_lag, X_test_lag = X_lag.iloc[:split], X_lag.iloc[split:]
y_train_lag, y_test_lag = y_lag.iloc[:split], y_lag.iloc[split:]

gbr = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.06,
                                  subsample=0.8, random_state=42)
gbr.fit(X_train_lag, y_train_lag)
y_pred_gbr = gbr.predict(X_test_lag)

gbr_mae = mean_absolute_error(y_test_lag, y_pred_gbr)
gbr_mape = np.mean(np.abs((y_test_lag - y_pred_gbr) / y_test_lag)) * 100
print(f'GBR Forecast MAE:  {gbr_mae:.2f} visits | MAPE: {gbr_mape:.2f}%')

In [ ]:
# ─── 7. Ensemble Forecast + Visualization ─────────────────────────────────────
# Weighted ensemble (ARIMA 40% + GBR 60%)
ensemble_pred = 0.4 * forecast_mean.values + 0.6 * y_pred_gbr
ensemble_mae = mean_absolute_error(test.values[:len(ensemble_pred)], ensemble_pred)
ensemble_mape = np.mean(np.abs((test.values[:len(ensemble_pred)] - ensemble_pred) / test.values[:len(ensemble_pred)])) * 100

print('═'*55)
print('MODEL COMPARISON — 30-Day Holdout')
print('═'*55)
print(f'  ARIMA(2,1,2):          MAE={arima_mae:.2f} | MAPE={arima_mape:.2f}%')
print(f'  GBR (14 lags):         MAE={gbr_mae:.2f}  | MAPE={gbr_mape:.2f}%')
print(f'  Ensemble (40%/60%):    MAE={ensemble_mae:.2f}  | MAPE={ensemble_mape:.2f}% ← Best')

# Plot
plt.figure(figsize=(18, 6))
# Historical context (last 60 days)
plt.plot(ts[-60:].index, ts[-60:].values, color='#06b6d4', linewidth=2, label='Actual', alpha=0.9)

# Predictions
test_dates = test.index[:len(ensemble_pred)]
plt.plot(test_dates, forecast_mean.values, color='#8b5cf6', linewidth=2, linestyle='--', label=f'ARIMA (MAPE={arima_mape:.1f}%)')
plt.plot(test_dates, y_pred_gbr, color='#f59e0b', linewidth=2, linestyle=':', label=f'GBR (MAPE={gbr_mape:.1f}%)')
plt.plot(test_dates, ensemble_pred, color='#10b981', linewidth=2.5, label=f'Ensemble (MAPE={ensemble_mape:.1f}%)')

# Confidence interval from ARIMA
plt.fill_between(test_dates, conf_int.iloc[:,0].values, conf_int.iloc[:,1].values,
                  alpha=0.15, color='#8b5cf6', label='90% CI (ARIMA)')

plt.axvline(train.index[-1], color='white', linestyle='--', alpha=0.4, label='Train/Test split')
plt.title('ED Volume Forecast — ARIMA vs GBR vs Ensemble (30-Day Holdout)', fontsize=13, color='white')
plt.ylabel('Daily Visits', color='white')
plt.legend(loc='upper left', fontsize=9)
plt.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../docs/ts_fig4_forecast_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Forecast comparison saved')
print('\n✅ Time-Series Forecasting Notebook Complete!')